In [62]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from ultralytics import YOLO
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from skimage import measure, morphology

In [40]:

train_image_path = os.path.join("..", "data","train","img")
val_image_path = os.path.join("..", "data","val","img")
test_image_path = os.path.join("..", "data","test","img")
train_mask_path = os.path.join("..", "data","train","mask")
val_mask_path = os.path.join("..", "data","val","mask")
test_mask_path = os.path.join("..", "data","test","mask")

In [41]:
# Get sorted lists of file paths
train_image_files = sorted(glob.glob(os.path.join(train_image_path, "*.tif")))
train_mask_files = sorted(glob.glob(os.path.join(train_mask_path, "*.tif")))
val_image_files = sorted(glob.glob(os.path.join(val_image_path, "*.tif")))
val_mask_files = sorted(glob.glob(os.path.join(val_mask_path,"*.tif")))
test_image_files = sorted(glob.glob(os.path.join(test_image_path, "*.tif")))
test_mask_files = sorted(glob.glob(os.path.join(test_mask_path,"*.tif")))



In [55]:
train_image = []
train_mask = []
for img_path, mask_path in zip(train_image_files, train_mask_files):

    #load the files
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    mask = np.array(cv2.imread(mask_path, cv2.IMREAD_UNCHANGED))

    train_image.append(img)
    train_mask.append(mask)

val_image = []
val_mask = []
for img_path, mask_path in zip(val_image_files, val_mask_files):

    #load the files
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    val_image.append(img)
    val_mask.append(mask)

test_image = []
test_mask = []
for img_path, mask_path in zip(test_image_files, test_mask_files):

    #load the files
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)


    test_image.append(img)
    test_mask.append(mask)


In [64]:
# Erode the mask to make trees separate more
def process_mask(mask):
    binary_mask = mask.copy()
    binary_mask = morphology.binary_erosion(binary_mask, footprint=np.ones((6,6)))
    return binary_mask

def extract_bbox(mask):
    binary_mask = mask.copy()

    labels = measure.label(binary_mask, connectivity=2)
    region_props = measure.regionprops(labels)

    bboxes = []
    centers = []
    for region in region_props:
        bbox = region.bbox # returns: (min_row, min_col, max_row, max_col)
        center = region.centroid # return (row, col)
        bboxes.append(bbox)
        centers.append(center)
    return bboxes

In [81]:
# Define augmentation pipeline
transform = A.Compose([
    A.RandomCrop(width=512, height=512),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussNoise(p=0.2),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))


In [68]:
def convert_to_yolo_format(bboxes, img_width, img_height):
    """ Convert Pascal VOC format (x_min, y_min, x_max, y_max) to YOLO format """
    yolo_bboxes = []
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox
        x_center = (x_min + x_max) / 2 / img_width
        y_center = (y_min + y_max) / 2 / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height
        yolo_bboxes.append([0, x_center, y_center, width, height])  # Assuming class_id = 0 for trees
    return yolo_bboxes

In [70]:
# extract my bbox in pascal voc format
train_mask_processed = []
for mask in train_mask:
    new_mask = process_mask(mask)
    train_mask_processed.append(new_mask)
train_bbox = []
for mask in train_mask_processed:
    bbox = extract_bbox(mask)
    train_bbox.append(bbox)


In [82]:
# apply augmentation
augmented_train = []
for img, bbox in zip(train_image, train_bbox):
    if len(bbox) == 0:  # Handle images with no objects
        augmented_train.append({'image': img, 'bboxes': [], 'class_labels': []})
        continue

    augmented = transform(image=img, bboxes=bbox, class_labels=[0]*len(bbox))
    augmented_train.append(augmented)

In [60]:
#Train a model
model = YOLO("yolo11n.pt")  # Load a pretrained model
results = model.train(data=train_data_yolo, epochs=100, imgsz=640)